<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 7 · Interactividad y narrativa
**Analista de datos · Power BI + Tableau** — Detalle, información sobre herramientas y móvil · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/07_interactividad_y_narrativa.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-07)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Recorrer una ruta de detalle de la empresa al producto.
2. Diseñar la información que aparece al pasar el mouse.
3. Ver una página pensada para el celular.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿De dónde sale la venta? Empresa → sede → categoría → producto |
> | **Qué se grafica** | Gráfico solar interactivo: haz clic en un anillo para entrar al detalle y en el centro para volver |
> | **Por qué esta forma** | Representa la jerarquía completa y permite profundizar con clics, como la obtención de detalles. |
> | **Qué se descartó** | Cuatro tablas separadas: se pierde la relación entre niveles. |
> | **Cómo se lee** | Haz clic en una sede: el gráfico muestra solo sus categorías. Es la misma idea de una página de detalle filtrada. |
> | **En Excel 365** | En Power BI: jerarquía en el eje y botones de exploración (flecha hacia abajo), u obtención de detalles hacia otra página. |

In [ ]:
# @title Gráfico 1 · Ruta de detalle
v25 = validas[validas.anio == 2025]
fig = px.sunburst(v25, path=["ciudad", "categoria", "producto"], values="subtotal", color="ciudad",
                  color_discrete_sequence=[AZUL, TEAL, MAGENTA, "#7A86E8", "#E8A33D", "#5BAA6A"])
fig.update_traces(hovertemplate="<b>%{label}</b><br>Venta neta: $ %{value:,.0f}<br>%{percentParent:.1%} del nivel superior<extra></extra>")
fig.update_layout(title="Venta neta 2025: haz clic en una sede para ver sus categorías y productos", height=560)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué conviene mostrar al pasar el mouse? |
> | **Qué se grafica** | Barras de venta por sede con información sobre herramientas: meta, cumplimiento, facturas y ticket |
> | **Por qué esta forma** | La barra responde la pregunta principal; la información sobre herramientas da el contexto sin ocupar espacio. |
> | **Qué se descartó** | Poner todas las cifras como etiquetas: la página se llena de números. |
> | **Cómo se lee** | Pasa el mouse por cada barra: ves las cuatro cifras que el gerente pediría a continuación. |
> | **En Excel 365** | Página de información sobre herramientas con tres o cuatro tarjetas, asignada en Formato › Información sobre herramientas. |

In [ ]:
# @title Gráfico 2 · Información sobre herramientas
met = metas[metas.anio == 2025].merge(sedes[["id_sede", "ciudad"]], on="id_sede").groupby("ciudad")["meta_subtotal"].sum()
t = v25.groupby("ciudad").agg(venta=("subtotal", "sum"), facturas=("id_factura", "nunique")).join(met).sort_values("venta")
t["cumpl"], t["ticket"] = t.venta / t.meta_subtotal, t.venta / t.facturas
fig = go.Figure(go.Bar(x=t.venta / 1e6, y=t.index, orientation="h", marker_color=[MAGENTA if c < 1 else GRIS for c in t.cumpl],
                       customdata=np.c_[t.meta_subtotal / 1e6, t.cumpl * 100, t.facturas, t.ticket],
                       hovertemplate="<b>%{y}</b><br>Venta: $ %{x:,.0f} M<br>Meta: $ %{customdata[0]:,.0f} M<br>Cumplimiento: %{customdata[1]:.1f} %<br>Facturas: %{customdata[2]:,.0f}<br>Ticket: $ %{customdata[3]:,.0f}<extra></extra>"))
fig.update_layout(title="Pasa el mouse por una barra: magenta son las sedes debajo de la meta", xaxis_title=MM, height=360)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo se ve el detalle de todas las sedes en el celular? |
> | **Qué se grafica** | Pequeños múltiplos: venta mensual 2025 de cada sede, en dos columnas angostas |
> | **Por qué esta forma** | Mismos ejes para todas: se comparan las formas de un vistazo y caben en una pantalla vertical. |
> | **Qué se descartó** | Un gráfico con seis líneas encimadas: ilegible en el celular. |
> | **Cómo se lee** | Cada panel es una sede. En Power BI, el diseño para móvil apila los visuales en una columna. |
> | **En Excel 365** | Vista › Diseño para móvil; en el informe, el visual con «Pequeños múltiplos» (Sedes[ciudad]). |

In [ ]:
# @title Gráfico 3 · Pequeños múltiplos por sede
m = v25.groupby(["ciudad", "mes"])["subtotal"].sum().reset_index()
m["Mes"] = m.mes.map(lambda x: MESES[x - 1]); m["Millones"] = m.subtotal / 1e6
fig = px.line(m, x="Mes", y="Millones", facet_col="ciudad", facet_col_wrap=2, category_orders={"ciudad": ORDEN_CIUDADES, "Mes": MESES},
              color_discrete_sequence=[AZUL])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(title="Venta mensual 2025 por sede (escala común)", height=620, width=520)
fig.show()

## Retos de interpretación

In [ ]:
# @title Reto 1 · En el gráfico 1, ¿qué categoría vende más en Barranquilla?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Papelería", "Tecnología", "Aseo y cafetería", "Mobiliario", "Impresión"]
revisar(respuesta, v25[v25.ciudad == "Barranquilla"].groupby("categoria")["subtotal"].sum().idxmax(), """Haz clic en Barranquilla y mira el anillo más grande.""")

In [ ]:
# @title Reto 2 · ¿Qué herramienta de Power BI lleva de una barra a una página de detalle filtrada?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Marcador", "Obtención de detalles", "Segmentación", "Información sobre herramientas"]
revisar(respuesta, "Obtención de detalles", """Una sola página de detalle recibe el filtro desde cualquier visual.""")

In [ ]:
# @title Reto 3 · ¿Cuál es el equivalente de los marcadores en Tableau?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Hojas", "Historias", "Filtros", "Parámetros"]
revisar(respuesta, "Historias", """Una historia es una secuencia de vistas para contar un hallazgo.""")

## Apóyate en IA
Para el guion de una presentación con marcadores:

```text
Mi tablero muestra [RESUMEN]. El hallazgo principal es [HALLAZGO].
Propón tres marcadores (qué muestra cada uno y una frase de texto) para presentarlo en dos minutos.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* La obtención de detalles reemplaza las páginas repetidas.
* La información sobre herramientas da contexto sin llenar la página.
* Los marcadores cuentan la historia en pasos; el diseño para móvil la lleva al celular.

**Siguiente:** [Clave 8 · Publicar y portafolio](https://omarli-360.github.io/powerbi-tableau-analista/#clave-08) · [Cuaderno 08](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/08_publicar_y_gobernar.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>